# Runner profiles

Runs after `02_cleaning.ipynb`. Reads `../data/clean.csv`, writes two figures to `../figures/`.

Research question: which distinct runner profiles emerge from the available variables?

Same rule as the cleaning notebook. Nothing is asserted that the diagnostics do not support,
and the diagnostics run before the model, not after it.

In [ ]:
import os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

RS = 42
rng = np.random.default_rng(RS)

df = pd.read_csv('../data/clean.csv')
df.shape

## Features: `BODY`, unchanged

`BODY` from `02` is the right list. It already excludes `long_run_distance_km` (81% one value),
the spoilers, and the behaviour block. Two notes before using it.

`rest_days_per_week` is not just correlated with `runs_per_week`, it is `min(7 - runs_per_week, 3)`
for all 80,000 rows. Keeping both gives training frequency double weight in the distance metric.
That is a choice rather than a neutral default, so it is tested in the sensitivity section below.
It is kept here for continuity with `02`; dropping it is equally defensible.

`BEHAVIOUR` is left out entirely: none of those 15 columns moves finish time (section 3).

In [ ]:
BODY = ['age', 'running_experience_months', 'previous_marathon_count', 'weekly_mileage_miles',
        'runs_per_week', 'speed_work_sessions_per_week', 'rest_days_per_week',
        'cross_training_hours_per_week', 'resting_heart_rate_bpm', 'vo2_max', 'bmi',
        'injury_count']

# the identity that makes the two frequency columns one column
(df.rest_days_per_week == np.minimum(7 - df.runs_per_week, 3)).all()

In [ ]:
# finish time is held back. clustering on it would make the answer circular.
X = StandardScaler().fit_transform(df[BODY])
sub = X[rng.choice(len(X), 10_000, replace=False)]
X.shape

## Which variables carry any signal

Not a clustering step, but it decides what the profiles can possibly be made of.

In [ ]:
corr = (df.select_dtypes('number')
          .corrwith(df.actual_finish_time_minutes)
          .drop(['actual_finish_time_minutes', 'target_finish_time_minutes',
                 'goal_gap', 'dnf', 'medal', 'medal_outcome'], errors='ignore')
          .dropna())
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

print(f'{(corr.abs() < 0.06).sum()} of {len(corr)} numeric columns are flat against finish time')
corr.round(3).head(12)

The whole adherence and wellness block sits at |r| < 0.06. In a real dataset that would be a
finding worth a paragraph. Here it is evidence the generator drew those columns independently,
and it belongs in the limitations.

## Does the data cluster at all

Three checks, in order: is the space non-uniform, are the clusters compact, are they reproducible.
The third one matters most on synthetic data, where k-means will happily partition a single blob.

In [ ]:
def hopkins(X, n=1000, seed=0):
    """~0.5 = uniform, no clustering tendency. >0.75 = structure worth looking for."""
    rs = np.random.default_rng(seed)
    idx = rs.choice(len(X), n, replace=False)
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    w = nn.kneighbors(X[idx], return_distance=True)[0][:, 1]
    syn = rs.uniform(X.min(0), X.max(0), size=(n, X.shape[1]))
    u = nn.kneighbors(syn, n_neighbors=1, return_distance=True)[0][:, 0]
    return u.sum() / (u.sum() + w.sum())

[round(hopkins(X, seed=s), 3) for s in (0, 1, 2)]

In [ ]:
def split_half_ari(X, k, n_rep=8):
    """cluster two random halves, compare where they put the full set."""
    out = []
    for i in range(n_rep):
        idx = rng.permutation(len(X))
        a, b = X[idx[:len(X)//2]], X[idx[len(X)//2:]]
        ka = KMeans(k, n_init=10, random_state=i).fit(a)
        kb = KMeans(k, n_init=10, random_state=i + 99).fit(b)
        out.append(adjusted_rand_score(ka.predict(X), kb.predict(X)))
    return float(np.mean(out))

ks = list(range(2, 9))
diag = pd.DataFrame({
    'silhouette': [silhouette_score(sub, KMeans(k, n_init=10, random_state=RS).fit_predict(sub)) for k in ks],
    'split_half_ari': [split_half_ari(sub, k) for k in ks],
}, index=ks).round(3)
diag.index.name = 'k'
diag

Silhouette never clears 0.15, so there are no compact separated groups anywhere in this space.
Stability holds through k=5 and drops below the threshold at k=6 and k=7. k=4 is the largest
partition that both reproduces and can be described in a sentence each, so that is the one taken
forward.

Stated plainly for the report: these are cut points on a continuum, not discovered types.

## Choice of algorithm

The diagnostics above say there are no density-separated groups, only a continuous cloud. That
makes this a partitioning problem rather than a discovery problem, and the algorithm should be
chosen on that basis rather than by habit. Nine candidates, same standardised matrix, k=4 where k
is a parameter.

Judge on `eta2`, not `silhouette`. Silhouette rewards a solution that shaves off a handful of
outliers and calls the remaining blob a cluster, which is exactly what average linkage does
here.

In [ ]:
import warnings
from sklearn.cluster import AgglomerativeClustering, HDBSCAN, Birch, SpectralClustering
from sklearn.mixture import GaussianMixture

K = 4
ok_mask = (df.dnf == 0).values
sub_idx = rng.choice(len(X), 10_000, replace=False)
bench_X, bench_ok = X[sub_idx], ok_mask[sub_idx]
bench_t = df.actual_finish_time_minutes.values[sub_idx]


def score(name, lab):
    """eta^2 on finish time, silhouette, and how lopsided the partition is."""
    keep = lab >= 0
    n_k = len(set(lab[keep]))
    s = pd.Series(bench_t[bench_ok & keep])
    gm = s.mean()
    b = s.groupby(pd.Series(lab[bench_ok & keep])).apply(lambda x: len(x) * (x.mean() - gm) ** 2).sum()
    sizes = np.bincount(lab[keep])
    return {'algorithm': name, 'k': n_k, 'noise': round((lab == -1).mean(), 3),
            'smallest_cluster': round(sizes.min() / keep.sum(), 3),
            'silhouette': round(silhouette_score(bench_X[keep], lab[keep]), 3) if n_k > 1 else np.nan,
            'eta2': round(b / ((s - gm) ** 2).sum(), 3)}


with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    bench = [
        score('k-means', KMeans(K, n_init=10, random_state=RS).fit_predict(bench_X)),
        score('GMM spherical', GaussianMixture(K, covariance_type='spherical', n_init=3, random_state=RS).fit_predict(bench_X)),
        score('GMM diagonal', GaussianMixture(K, covariance_type='diag', n_init=3, random_state=RS).fit_predict(bench_X)),
        score('GMM full', GaussianMixture(K, covariance_type='full', n_init=3, random_state=RS).fit_predict(bench_X)),
        score('agglomerative ward', AgglomerativeClustering(K, linkage='ward').fit_predict(bench_X)),
        score('agglomerative complete', AgglomerativeClustering(K, linkage='complete').fit_predict(bench_X)),
        score('agglomerative average', AgglomerativeClustering(K, linkage='average').fit_predict(bench_X)),
        score('Birch', Birch(n_clusters=K).fit_predict(bench_X)),
        score('spectral (kNN)', SpectralClustering(K, affinity='nearest_neighbors', n_neighbors=20,
                                                   assign_labels='kmeans', random_state=RS).fit_predict(bench_X)),
    ]
    for mcs in (100, 500, 1000):
        lab = HDBSCAN(min_cluster_size=mcs).fit_predict(bench_X)
        bench.append(score(f'HDBSCAN min_size={mcs}', lab) if (lab >= 0).any() else
                     {'algorithm': f'HDBSCAN min_size={mcs}', 'k': 0, 'noise': 1.0,
                      'smallest_cluster': np.nan, 'silhouette': np.nan, 'eta2': np.nan})

pd.DataFrame(bench).set_index('algorithm').sort_values('eta2', ascending=False)

In [ ]:
# stability, for the three that can label unseen rows
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    stab = {
        'k-means': split_half_ari(bench_X, K),
        'GMM spherical': np.mean([adjusted_rand_score(
            *[GaussianMixture(K, covariance_type='spherical', n_init=3, random_state=r).fit(
                bench_X[p]).predict(bench_X) for p, r in
              [(slice(None, 5000), i), (slice(5000, None), i + 99)]])
            for i in range(6)]),
        'GMM full': np.mean([adjusted_rand_score(
            *[GaussianMixture(K, covariance_type='full', n_init=3, random_state=r).fit(
                bench_X[p]).predict(bench_X) for p, r in
              [(slice(None, 5000), i), (slice(5000, None), i + 99)]])
            for i in range(6)]),
    }
print(pd.Series(stab).round(3).rename('split_half_ari').to_string())

# do a centroid method and an agglomerative method agree on where the boundaries go?
km_lab = KMeans(K, n_init=10, random_state=RS).fit_predict(bench_X)
ward_lab = AgglomerativeClustering(K, linkage='ward').fit_predict(bench_X)
round(adjusted_rand_score(km_lab, ward_lab), 3)

k-means wins on eta^2 by a wide margin, and is the only method with split-half ARI above 0.7.

The failures are informative rather than incidental. HDBSCAN returns 100% noise at any sensible
`min_cluster_size`, which is the cleanest possible confirmation that there is nothing
density-separated to find. Full-covariance GMM has the freedom to fit elongated overlapping
ellipses and spends it on noise: lowest silhouette in the table and split-half ARI near 0.41
against 0.93 for k-means. Average linkage produces the best silhouette of any k=4 partition and
close to the worst eta^2, by isolating a handful of outliers and calling the rest one cluster,
which is why `smallest_cluster` is in the table.

The Ward comparison does **not** come out the way a robustness check is supposed to. Ward and
k-means agree that four groups exist and disagree substantially about which runners go in them:
ARI 0.36 between the two labelings, and Ward recovers under half the finish-time signal. That is
not a reason to distrust the k-means solution so much as another statement of the same underlying
fact. With no natural boundaries in the cloud, where the boundaries land depends on the rule used
to draw them. The k-means partition is stable under resampling of its own rule (ARI 0.93) and it
is the one that tracks performance best, but it is not the only defensible cut of this data, and
the report should say so rather than present four profiles as though they were discovered.

With that stated, k-means is still the right choice rather than a default. The data is one
continuous cloud with a few dominant axes, the task is cutting it into interpretable regions, and
k-means' rigidity is what stops it chasing noise. Methods built to discover irregular
density-separated groups have no advantage where there are none.

If `gender` and `training_program` are added later, k-means stops being valid on one-hot columns:
switch to k-prototypes (`kmodes`) or Gower distance with PAM.

## The four profiles

In [ ]:
K = 4
df['profile'] = KMeans(K, n_init=10, random_state=RS).fit_predict(X)

# relabel slowest -> fastest so the numbering means something across reruns
order = df.groupby('profile').actual_finish_time_minutes.mean().sort_values(ascending=False).index
df['profile'] = df.profile.map({old: new for new, old in enumerate(order)})

NAMES = ['Low-frequency\nbeginners', 'Frequent\nshort runs',
         'High aerobic\ncapacity', 'Experienced\nhigh-load']
shares = df.profile.value_counts(normalize=True).sort_index()
shares.round(3)

In [ ]:
# z-deviations are what makes the profiles readable; raw means are what goes in the report table
z = df.groupby('profile')[BODY].mean().sub(df[BODY].mean()).div(df[BODY].std())
z.T.round(2)

In [ ]:
df.groupby('profile')[BODY].mean().T.round(1)

Read down the columns:

0. **Low-frequency beginners**, 44%. 3.7 runs a week, 3 rest days, 21 months of running, 70% on the beginner programme.
1. **Frequent short runs**, 25%. 5.2 runs a week on the same 16 miles as group 0, physiology unremarkable.
2. **High aerobic capacity**, 16%. VO2 max 65 against 54 overall, resting HR 65, youngest group, training only moderate.
3. **Experienced high-load**, 15%. 95 months, 22.5 miles, 1.6 speed sessions, 85% on the advanced programme.

Groups 0 and 1 differ almost entirely on frequency, which is the dimension `rest_days_per_week`
doubles. That is the visible cost of keeping it, and the sensitivity check below shows what
happens without it.

## Validation against outcomes

Finish time was never in the feature space, so this is a real test rather than a restatement.

In [ ]:
ok = df[df.dnf == 0]
val = ok.groupby('profile').actual_finish_time_minutes.agg(['mean', 'std', 'count']).round(1)

g = ok.actual_finish_time_minutes.mean()
between = ok.groupby('profile').actual_finish_time_minutes.apply(lambda s: len(s) * (s.mean() - g) ** 2).sum()
eta2 = between / ((ok.actual_finish_time_minutes - g) ** 2).sum()

print(f'eta^2 = {eta2:.3f}')
val

In [ ]:
# two things the model never saw: drop-out rate and programme mix
pd.concat([df.groupby('profile').dnf.mean().round(3).rename('dnf_rate'),
           pd.crosstab(df.profile, df.training_program, normalize='index').round(2)], axis=1)

51 minutes between the slowest and fastest profile, 27% of finish-time variance, and a drop-out
rate that more than doubles from the fastest group to the slowest. The programme mix moves from
70% beginner to 85% advanced without the programme ever being a feature. The profiles are weakly
separated but they are not arbitrary.

## Sensitivity: does the answer depend on the two contested columns

`rest_days_per_week` in or out, `long_run_distance_km` in or out. Three feature sets, same pipeline.

In [ ]:
sets = {
    'BODY (as used)':        BODY,
    'BODY - rest_days':      [c for c in BODY if c != 'rest_days_per_week'],
    'BODY - rest_days + long_run': [c for c in BODY if c != 'rest_days_per_week'] + ['long_run_distance_km'],
}

rows = []
for name, cols in sets.items():
    Xs = StandardScaler().fit_transform(df[cols])
    ss = Xs[rng.choice(len(Xs), 8000, replace=False)]
    lab = KMeans(K, n_init=10, random_state=RS).fit_predict(Xs)
    t = ok.actual_finish_time_minutes
    gr = pd.Series(lab, index=df.index).loc[ok.index]
    b = t.groupby(gr).apply(lambda s: len(s) * (s.mean() - g) ** 2).sum()
    rows.append({'features': name, 'n_vars': len(cols),
                 'silhouette': round(silhouette_score(ss, KMeans(K, n_init=10, random_state=RS).fit_predict(ss)), 3),
                 'split_half_ari': round(split_half_ari(ss, K), 3),
                 'eta2': round(b / ((t - g) ** 2).sum(), 3)})

pd.DataFrame(rows).set_index('features')

All three sets land in the same place: silhouette in the low 0.1s, stable at k=4, eta^2 within
0.01 of each other. Neither contested column changes the answer, which is the useful result here.

The one substantive difference is in the composition rather than the metrics. Putting
`long_run_distance_km` back produces a group defined by that column alone at +2.8 SD, which is
simply the 19% of rows not sitting on the floor value. That is an artefact of the column, not a
kind of runner, and the decision in `02` to leave it out is what prevents it appearing. Worth one
sentence in the report.

In [ ]:
# the excluded column is flat across the profiles, which is the check that the exclusion cost nothing
df.groupby('profile').long_run_distance_km.mean().round(1)

## Figures

In [ ]:
plt.rcParams.update({'figure.dpi': 150, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.titlesize': 9, 'axes.titleweight': 'bold',
                     'axes.labelsize': 8, 'legend.frameon': False})
INK, ACC, GREY = '#2b2b2b', '#c0392b', '#9aa0a6'
PAL = ['#4c6ef5', '#e8590c', '#2f9e44', '#ae3ec9']

os.makedirs('../figures', exist_ok=True)

pca_full = PCA().fit(X)
cum = np.cumsum(pca_full.explained_variance_ratio_)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(11, 6.2))

a = ax[0, 0]
a.hist(df.long_run_distance_km, bins=np.arange(14.5, 42, 0.5), color=GREY)
a.set_title('Why long_run stays out'); a.set_xlabel('long_run_distance_km'); a.set_ylabel('runners')
a.annotate(f'{(df.long_run_distance_km == 15.0).sum():,} rows at exactly 15.0 km',
           xy=(15.4, 55000), xytext=(21, 50000), color=ACC, fontsize=7.5,
           arrowprops=dict(arrowstyle='->', color=ACC, lw=0.9))

a = ax[0, 1]
a.hist(df.weekly_mileage_miles, bins=np.arange(12, 61, 1), color=GREY)
a.set_title('Weekly mileage has the same floor'); a.set_xlabel('weekly_mileage_miles')
a.annotate(f'{(df.weekly_mileage_miles == 12.4).sum():,} rows at exactly 12.4 mi',
           xy=(13, 38000), xytext=(24, 33000), color=ACC, fontsize=7.5,
           arrowprops=dict(arrowstyle='->', color=ACC, lw=0.9))

a = ax[0, 2]
top = corr[:16][::-1]
a.barh(top.index, top.values, color=[ACC if abs(v) >= 0.1 else GREY for v in top.values])
a.axvline(0, color=INK, lw=0.7)
a.set_title('Correlation with finish time'); a.set_xlabel('Pearson r'); a.tick_params(labelsize=6.5)

a = ax[1, 0]
a.plot(range(1, len(cum) + 1), cum, 'o-', color=INK, ms=3.5, lw=1.2)
a.axhline(0.85, color=ACC, ls='--', lw=0.9); a.text(6.2, 0.865, '85% of variance', color=ACC, fontsize=7)
a.set_title('PCA: variance is spread thin'); a.set_xlabel('components')
a.set_ylabel('cumulative variance'); a.set_ylim(0, 1.02)

a = ax[1, 1]
a.plot(ks, diag.silhouette, 'o-', color=INK, ms=4, lw=1.3)
a.axhspan(0, 0.25, color=ACC, alpha=0.07)
a.text(4.4, 0.045, 'below 0.25 = no meaningful separation', color=ACC, fontsize=7)
a.set_ylim(0, 0.6); a.set_title('Silhouette by k'); a.set_xlabel('k'); a.set_ylabel('mean silhouette')

a = ax[1, 2]
a.plot(ks, diag.split_half_ari, 'o-', color=INK, ms=4, lw=1.3)
a.axhline(0.75, color=ACC, ls='--', lw=0.9); a.text(2.1, 0.66, 'stability threshold', color=ACC, fontsize=7)
a.axvline(K, color=PAL[0], lw=6, alpha=0.15)
a.set_ylim(0, 1.02); a.set_title('Split-half stability (ARI)'); a.set_xlabel('k')
a.set_ylabel('adjusted Rand index')

fig.suptitle('Data quality and clustering diagnostics', fontsize=11, fontweight='bold',
             x=0.02, ha='left', y=0.985)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig('../figures/fig1_diagnostics.png', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(11, 7.2))
gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1], hspace=0.42, wspace=0.22)

a = fig.add_subplot(gs[0, :])
m = z.T.values
im = a.imshow(m, cmap='RdBu_r', vmin=-2.0, vmax=2.0, aspect='auto')
a.set_xticks(range(K)); a.set_xticklabels([f'{NAMES[i]}\n({shares[i]*100:.0f}%)' for i in range(K)], fontsize=7.5)
a.set_yticks(range(len(BODY))); a.set_yticklabels(BODY, fontsize=7)
for i in range(len(BODY)):
    for j in range(K):
        a.text(j, i, f'{m[i, j]:+.1f}', ha='center', va='center', fontsize=6.5,
               color='white' if abs(m[i, j]) > 1.2 else INK)
a.set_title('Profile signatures (standard deviations from the overall mean)')
fig.colorbar(im, ax=a, shrink=0.8, pad=0.01).set_label('z-score', fontsize=7)

P = PCA(n_components=2).fit(X)
XY = P.transform(X)
a = fig.add_subplot(gs[1, 0])
s = rng.choice(len(XY), 6000, replace=False)
a.scatter(XY[s, 0], XY[s, 1], c=[PAL[i] for i in df.profile.values[s]], s=3, alpha=0.35, linewidths=0)
a.set_title('The profiles overlap: no natural gaps')
a.set_xlabel(f'PC1 ({P.explained_variance_ratio_[0]*100:.1f}%)')
a.set_ylabel(f'PC2 ({P.explained_variance_ratio_[1]*100:.1f}%)')
a.legend(handles=[Line2D([], [], marker='o', ls='', color=PAL[i], ms=5,
                         label=NAMES[i].replace('\n', ' ')) for i in range(K)], fontsize=6.5, loc='upper left')

a = fig.add_subplot(gs[1, 1])
data = [ok.loc[ok.profile == i, 'actual_finish_time_minutes'].values for i in range(K)]
parts = a.violinplot(data, vert=False, showmedians=True, widths=0.85)
for pc, c in zip(parts['bodies'], PAL):
    pc.set_facecolor(c); pc.set_alpha(0.55); pc.set_edgecolor('none')
for key in ('cmedians', 'cbars', 'cmins', 'cmaxes'):
    parts[key].set_color(INK); parts[key].set_linewidth(0.9)
a.set_yticks(range(1, K + 1))
a.set_yticklabels([f"{NAMES[i].replace(chr(10), ' ')}\n{np.mean(data[i]):.0f} min" for i in range(K)], fontsize=7)
a.set_xlabel('actual finish time (minutes)'); a.set_title(f'Finish time by profile  (eta² = {eta2:.2f})')

fig.suptitle(f'Four runner profiles: k = {K}, silhouette {diag.silhouette[K]:.2f}, '
             f'split-half ARI {diag.split_half_ari[K]:.2f}, eta² = {eta2:.2f}',
             fontsize=11, fontweight='bold', x=0.02, ha='left', y=0.98)
fig.savefig('../figures/fig2_profiles.png', bbox_inches='tight')

## Save

In [ ]:
df[['runner_id', 'profile']].to_csv('../data/profiles.csv', index=False)
df.profile.value_counts().sort_index()

Four profiles, reproducible at k=4 and not below, explaining 27% of finish-time variance and
51 minutes end to end. Weakly separated throughout: silhouette 0.12, and the PCA scatter shows a
gradient rather than four groups. A different partitioning rule cuts the same cloud in a
different place (Ward against k-means, ARI 0.36), so the boundaries are a choice and not a
discovery. Reported as regions of a continuum, which is what the diagnostics support.